# Basic OLS

This notebook estimates a linear regression and reports traditional standard errors (assuming iid residuals).

For a package, consider [GLM.jl](https://github.com/JuliaStats/GLM.jl) or [LinearRegression.jl](https://github.com/st--/LinearRegression.jl) (not used here).

## Load Packages and Extra Functions

The key functions are from the `FinEcmt_OLS` module found in the `src` subfolder.

The `DelimitedFiles` package is used for importing the csv data file and the `LinearAlgebra` package for some matrix operations (eg. `diag()`, which extracts the diagonal of a matrix.)

In [1]:
MyModulePath = joinpath(pwd(),"src")     #add /src to module path
!in(MyModulePath,LOAD_PATH) && push!(LOAD_PATH,MyModulePath)
using FinEcmt_OLS

In [2]:
#=
include(joinpath(pwd(),"src","FinEcmt_OLS.jl"))
using .FinEcmt_OLS
=#

In [3]:
using Statistics, DelimitedFiles, LinearAlgebra

## Loading Data

In [4]:
x = readdlm("Data/TwoIndustries.csv",',',skipstart=1)

                #yearmonth, 2 industries, factors
(dN,Re,F) = (x[:,1],Float64.(x[:,2:3]),Float64.(x[:,4:end]))   #make sure data is Float
x = nothing
printlnPs("Re: ",size(Re),"\n","F: ",size(F))

y = Re[:,1]                    #to get standard OLS notation, here one return series
T = size(y,1)
x = [ones(T) F]
k = size(x,2)

println("\nT and k: $T $k")

      Re:   (660, 2)          
       F:   (660, 3)

T and k: 660 4


## OLS Estimates and Their Distribution

Consider the linear regression

$y_{t}=\beta^{\prime}x_{t}+u_{t},$

When $x_t$ and $u_t$ are independent and $u_t$ is iid (Gauss-Markov assumptions), then the distribution of the estimates is (typically)

$\hat{\beta} \sim  N(\beta_{0},S_{xx}^{-1}\sigma^2),$

where $\sigma^2$ is the variance of the residual and $S_{xx} = \sum\nolimits_{t=1}^{T}x_{t}x_{t}^{\prime}$. Whether $\sigma^2$ is estimated using $1/T$ or $1/(T-1)$ affects the results slightly.

In matrix form, these expressions are 
$Y = Xb + u, \: \text{ and } \: S_{xx} = X'X,$
where $X$ is defined below.


### Matrix Form

To calculate the estimates it is often convenient to work with matrices. Define $X_{T\times k}$ by 
letting $x_{t}^{\prime}$ and be the $t^{th}$ row

$$
X_{T\times k}=
\begin{bmatrix}
x_{1}^{\prime}\\
\vdots\\
x_{T}^{\prime}
\end{bmatrix}
$$

In contrast $Y$ is a just a vector with $T$ elements (or possibly a $T \times 1$ matrix).

This is implemented in the `OlsGM()` function from the `FinEcmt_OLS` module. The source file is in the `src` subfolder. The next cells print the documentation and source code of the function. In particular, notice the `b = X\Y` and `V = inv(X'X)*σ²`.

In [5]:
@doc2 OlsGM

```julia
OlsGM(Y,X)
```

LS of Y on X; for one dependent variable, Gauss-Markov assumptions

### Input

  * `Y::Vector`:    T-vector, the dependent variable
  * `X::Matrix`:    Txk matrix of regressors (including deterministic ones)

### Output

  * `b::Vector`:    k-vector, regression coefficients
  * `u::Vector`:    T-vector, residuals Y - yhat
  * `Yhat::Vector`: T-vector, fitted values X*b
  * `V::Matrix`:    kxk matrix, covariance matrix of b
  * `R²::Number`:   scalar, R² value


In [6]:
using CodeTracking
println(@code_string OlsGM([1],[1]))    #print the source code

function OlsGM(Y,X)

    T    = size(Y,1)

    b    = X\Y
    Yhat = X*b
    u    = Y - Yhat

    σ²   = var(u;corrected=false)        #using 1/T, not 1/(T-1)
    V    = inv(X'X)*σ²
    R²   = 1 - var(u)/var(Y)

    return b, u, Yhat, V, R²

end


## OLS Regression

In [7]:
(b,_,_,V,R²) = OlsGM(y,x)    
Stdb = sqrt.(diag(V))        #standard errors
       
printblue("OLS Results:\n")
xNames = ["c","Market","SMB","HML"]
printmat(b,Stdb,colNames=["b","std"],rowNames=xNames)

printlnPs("R²: ",R²)

OLS Results:

               b       std
c          0.142     0.109
Market     1.130     0.025
SMB        0.180     0.036
HML       -0.510     0.036

      R²:      0.823


In [8]:
RegressionTable(b,V,xNames)     #a function for printing regression results

            coef    stderr    t-stat   p-value
c          0.142     0.109     1.303     0.193
Market     1.130     0.025    45.854     0.000
SMB        0.180     0.036     4.939     0.000
HML       -0.510     0.036   -14.350     0.000



# Missing Values (extra)

The next cells use a simple function (`excise()`) to remove observations ($t$) where $y_t$ and/or some of the $x_t$ variables are NaN/missing. We illustrate the usage by a very simple example.

An alternative approach is to fill *both* $y_t$ and $x_t$ with zeros (if any of them contains NaN/missing) by using the `OLSyxReplaceNaN` function and then do the regression. This is illustrated in the subsequent cell.

In [9]:
(y0,x0) = (copy(y),copy(x))    #so we can can change some values
x0[2,2] = NaN                  #set a value to NaN

(y1,x1) = excise(y0,x0)        #gettting rid of obs t if any in (y_t,y_t) is Nan/missing
println("obs 1-3 before")
printmat(y0[1:3],x0[1:3,:];colNames=vcat("y",xNames))

println("after")
printmat(y1[1:3],x1[1:3,:];colNames=vcat("y",xNames))

printblue("OLS using only observations without any NaN/missing:")
b = x1\y1
printmat(b)

obs 1-3 before
         y         c    Market       SMB       HML
    -9.660     1.000    -8.100     2.930     3.130
     2.320     1.000       NaN    -2.580     3.930
    -3.150     1.000    -1.060    -2.320     3.990

after
         y         c    Market       SMB       HML
    -9.660     1.000    -8.100     2.930     3.130
    -3.150     1.000    -1.060    -2.320     3.990
   -14.840     1.000   -11.000    -6.150     6.180

OLS using only observations without any NaN/missing:
     0.143
     1.131
     0.179
    -0.509



In [10]:
(vv,y2,x2) = OLSyxReplaceNaN(y0,x0)        #alternative approach: fill (y_t,x_t) with zeros

println("after")
printmat(y2[1:3],x2[1:3,:];colNames=vcat("y",xNames))

printblue("OLS from setting observations with any NaN/missing to 0:")
b = x2\y2
printmat(b)

after
         y         c    Market       SMB       HML
    -9.660     1.000    -8.100     2.930     3.130
     0.000     0.000     0.000     0.000     0.000
    -3.150     1.000    -1.060    -2.320     3.990

OLS from setting observations with any NaN/missing to 0:
     0.143
     1.131
     0.179
    -0.509



# Different Ways to Calculate OLS Estimates (extra)

Recall that OLS can be calculated as

$\hat{\beta} = S_{xx}^{-1}S_{xy}, \: \text{ where } \: 
S_{xx}      = \sum\nolimits_{t=1}^{T}x_{t}x_{t}^{\prime}
\: \text{ and } \:
S_{xy}      = \sum\nolimits_{t=1}^{T}x_{t}y_{t}.$

The next cell calculates the OLS estimates in three different ways: *(1)* a loop to create $S_{xx}$ and $S_{xy}$ followed by $S_{xx}^{-1}S_{xy}$; *(2)* $(X'X)^{-1}X'Y$; *(3)* and `X\Y`. They should give the same result in well-behaved data sets, but (3) is probably the most stable version.

The point of this is to get familiar with different approaches, and see that they represent the same.

In [11]:
printblue("Three different ways to calculate OLS estimates:")

k    = size(x,2)
Sxx = zeros(k,k)
Sxy = zeros(k,1)
for t = 1:T
    #local x_t, y_t            #local/global is needed in script
    #global Sxx, Sxy
    x_t = x[t,:]               #a vector
    y_t = y[t]
    Sxx = Sxx + x_t*x_t'     #kxk, same as Sxx += x_t*x_t'
    Sxy = Sxy + x_t*y_t      #kx1, same as Sxy += x_t*y_t
end
b1 = inv(Sxx)*Sxy            #OLS coeffs, version 1

b2 = inv(x'x)*x'y            #OLS coeffs, version 2

b3 = x\y                     #OLS coeffs, version 3

printmat(b1,b2,b3,colNames=["1","2","3"],rowNames=xNames)

Three different ways to calculate OLS estimates:
               1         2         3
c          0.142     0.142     0.142
Market     1.130     1.130     1.130
SMB        0.180     0.180     0.180
HML       -0.510    -0.510    -0.510

